# Fuel duty

Fuel duty is a per-litre excise duty on petrol and diesel. PolicyEngine UK models fuel duty on vehicle fuel only (i.e. petrol and diesel for road transport), not fuel used for heating or other purposes.

Parameters live in `policyengine_uk/parameters/gov/hmrc/fuel_duty/` and the household-level liability is computed in `policyengine_uk/variables/gov/hmrc/fuel_duty/fuel_duty.py`.

## How PolicyEngine computes fuel duty

For each household we compute

```
fuel_duty = (petrol_and_diesel_rate - rural_relief * in_relief_area) * (petrol_litres + diesel_litres)
```

where

- `petrol_litres` and `diesel_litres` are the household's annual road-fuel volumes (split into monthly values by the variable),
- `petrol_and_diesel_rate` is the headline UK fuel duty rate (£/litre),
- `rural_fuel_duty_relief` is the per-litre discount applied in eligible rural areas, and
- `in_rural_fuel_duty_relief_area` is an input that flags households in eligible postcodes.

## Headline fuel duty rate

The UK government usually updates the headline rate at Spring or Autumn budgets. The rate was frozen at £0.5795/litre between April 2011 and March 2022, when a 5p cut was introduced. That cut has since been extended at successive budgets. The table and chart below show the historical headline rate from PolicyEngine's parameter file.

In [8]:
from policyengine_uk import CountryTaxBenefitSystem
import plotly.express as px
import pandas as pd

system = CountryTaxBenefitSystem()
hmrc = system.parameters.gov.hmrc

df = pd.DataFrame()

df["Date of change"] = [
    parameter.instant_str for parameter in hmrc.fuel_duty.petrol_and_diesel.values_list
]
df["Fuel duty rate (£/litre)"] = [
    parameter.value for parameter in hmrc.fuel_duty.petrol_and_diesel.values_list
]
df.sort_values("Date of change", inplace=True)
df.set_index("Date of change")

,Fuel duty rate (£/litre)
Date of change,
2010-04-01,0.5719
2010-10-01,0.5819
2011-01-01,0.5895
2011-03-23,0.5795
2012-01-01,0.6097
2013-04-01,0.5795
2021-01-01,0.5795
2022-03-23,0.5295
2023-04-01,0.5295


In [14]:
import plotly.express as px

px.line(
    df,
    x="Date of change",
    y="Fuel duty rate (£/litre)",
    title="Fuel duty rate on petrol and diesel",
    color_discrete_sequence=["#2C6496"],
).update_layout(
    yaxis_title="Fuel duty rate per litre",
    yaxis_tickprefix="£",
    yaxis_tickformat=",.2f",
    xaxis_title="Date",
    xaxis_tickformat="%Y",
    height=600,
    width=800,
    template="plotly_white",
).update_xaxes(tickangle=45, tickfont={"size": 10}).update_traces(line_shape="hv")

## Rural Fuel Duty Relief

The Rural Fuel Duty Relief Scheme grants a 5p/litre discount on retail petrol and diesel sold in eligible remote postcodes (parts of the Highlands, Western Isles, Orkney, Shetland, the Isles of Scilly and a handful of others). PolicyEngine models this through the `in_rural_fuel_duty_relief_area` input flag at the household level; the discount is then subtracted from the headline rate inside the `fuel_duty` formula.

The relief rate itself is stored in `policyengine_uk/parameters/gov/hmrc/fuel_duty/rural_fuel_duty_relief.yaml` and has been £0.05/litre since the scheme's introduction in March 2012.

## References

- HMRC, [Hydrocarbon Oil Duties Act 1979](https://www.legislation.gov.uk/ukpga/1979/5/contents) - the primary statute establishing fuel duty.
- HMRC, [Rural fuel duty relief scheme - Notice 2001](https://www.gov.uk/guidance/rural-duty-relief-scheme-notice-2001).
- Hydrocarbon Oil (Mileage Allowance for Rural Petrol Filling Stations) [Regulations 2011 (SI 2011/2935)](https://www.legislation.gov.uk/uksi/2011/2935/contents/made).